## Ollama Multimodal Capabilities

Just like `stream=True`, the `ollama.generate()` function supports **many other parameters** for different tasks:
- You can provide **multiple inputs** (text + image).
- You can request **structured outputs** (e.g., JSON).
- You can give **system instructions** to guide the model.
- You can modify existing parameters for fine‑tuned control.

### Multimodal (Vision + Text)
- Some models have **vision capability** (they can process images).
- You can pass an image (encoded as base64) along with a text prompt.
- The model then generates a description or analysis of the image.

### Steps for Working with Images
1. Load the image from disk (`ipl.png` in this case).
2. Read the image as bytes.
3. Convert bytes to **base64 string** (since Ollama expects base64 encoding).
4. Pass the base64 string inside the `image` parameter of `ollama.generate`.
5. Provide a text prompt (e.g., `"describe the image"`).
6. The model returns a response describing the image.

In [6]:
import ollama
import base64


In [7]:
# Step 1: Define the image path
image_path = r"C:\Users\admin\Desktop\GenAI\ollama\ipl.png"

In [8]:
# Step 2: Read the image as bytes
with open(image_path, "rb") as f:
    image_bytes = f.read()

### Why Base64, "rb", and UTF-8?

#### Base64 Encoding
- Computers store images as **binary data** (bytes).
- APIs like Ollama expect the image to be passed as **text**, not raw binary.
- **Base64** is a way to convert binary data into a text string using only ASCII characters.
- This makes the image safe to transmit in JSON or over HTTP without corruption.
- So: we read the image → convert to base64 → send it as a string.

#### "rb" Mode in `open()`
- `"r"` = read mode (default).
- `"b"` = binary mode.
- `"rb"` = read the file as raw binary bytes.
- Images are not plain text, so we must read them in binary mode to capture the exact byte values.

#### UTF-8 Decoding
- After base64 encoding, we get a **bytes object**.
- `.decode("utf-8")` converts those bytes into a **Python string**.
- UTF-8 is the standard text encoding that can represent all characters safely.
- This ensures the base64 string is usable inside JSON and Python code.

In short:
- **rb** → read image as raw bytes.  
- **base64** → convert bytes into safe text.  
- **utf-8 decode** → turn that text into a proper Python string for the API.

In [12]:
# Step 3: Convert to base64 string
image_base64 = base64.b64encode(image_bytes).decode("utf-8")

In [13]:
# Step 4: Generate response using qwen3.5:0.8b (vision-capable model)
response = ollama.generate(
    model="qwen3.5:0.8b",
    prompt="Describe the image in detail",
    images=[image_base64]   # pass the base64-encoded image
)

In [14]:
# Step 5: Print the model's description
print("Model Response:")
print(response.response)

Model Response:
Based on the visual content, here is a detailed description of the image:

**Overall Composition**
The image is a promotional graphic for a cricket team event, featuring nine Indian players posing in a group shot. The background is a solid dark green with subtle gold decorative elements at the top left and bottom right corners, and there are gold dotted lines running horizontally across the middle of the image.

**Text and Branding**
*   **Main Title:** The top left corner features the text "TATA IPL 2026" in a bold, gold-colored font.
*   **Subtitle:** Directly beneath the main title is smaller text in gold reading "PLAYER TRADE UPDATES".
*   **Logo:** A smaller gold "TATA" logo is visible on the far left side, possibly indicating the sponsor or a previous team affiliation.

**Players (Back Row / Top Row)**
The players are arranged in the top row, with their faces hidden and wearing hats or headgear, facing forward. The arrangement is slightly staggered.
1.  **Far Left

### Multimodal with Multiple IPL Images

We now explore how Ollama can handle **multiple images together**.  
This is powerful because:
- It can **compare visuals** (e.g., trade updates vs. player vs. fans).
- It can **connect contexts** (league-level, player-level, and audience-level).
- It shows how multimodal AI can reason across different perspectives.

#### Steps:
1. Load each image in binary mode (`rb`).
2. Convert each to base64 string (`utf-8` decode).
3. Add them all to the `images` parameter.
4. Provide a prompt that asks the model to **compare, analyze, and connect** the images.


In [ ]:
import ollama
import base64

def encode_image(path):
    """Helper function to read and encode an image as base64 string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# Encode multiple IPL images
ipl_trade_image = encode_image(r"C:\Users\admin\Desktop\GenAI\ollama\ipl.png")
ipl_player_image = encode_image(r"C:\Users\admin\Desktop\GenAI\ollama\ipl 2.png")
ipl_fans_image   = encode_image(r"C:\Users\admin\Desktop\GenAI\ollama\ipl 3.png")

# Generate response using qwen3.5:0.8b (vision-capable model)
response = ollama.generate(
    model="qwen3.5:0.8b",
    prompt=(
        "You are analyzing three IPL visuals:\n"
        "1. Trade updates graphic\n"
        "2. A Chennai Super Kings player in action\n"
        "3. Fans cheering in the stadium\n\n"
        "Describe each image, then explain how they connect together to tell the story of IPL 2026. "
        "Make it engaging, as if you are narrating to a classroom audience."
    ),
    images=[ipl_trade_image, ipl_player_image, ipl_fans_image]
)

print("Model Response:")
print(response.response)


🔹 Model Response:
Here is a breakdown of the three images and how they work together to tell the story of the 2026 IPL series.

**1. The Trade Roster Graphic**
This image shows the team roster at the "Tata IPL 2026." The text at the bottom clarifies this is a "PLAYER TRADE UPDATES." We can see a mix of players in different kits:
*   **Blue & Pink Jerseys:** These represent the new lineup, including players from Indian teams like the KENNINOUS and potentially the Royal.
*   **Yellow Jerseys:** These are the traditional favorites. The team appears to be **Chennai Super Kings (CSK)**, evidenced by the "CSK" on the backs and the "Gull Etihad Airways" sponsor.
*   **Red & White Jerseys:** This is the **Rajasthan Royals**, with the prominent "Hillbilly" and "Heineken" logos.

**2. A Player in Action**
This image captures a moment from the middle of the action. We see a player in full cream kit (yellow jersey) looking at the ball with a ready stance.
*   **The Player:** This is a key player, 